In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

from scipy.signal import find_peaks
import json

In [ ]:
# Reads an image file from disk and returns it as RGB (unless grayscale=True is asked for).
def read_image(path, grayscale=False):
    read_mode = cv2.IMREAD_GRAYSCALE if grayscale else cv2.IMREAD_COLOR  # pick how OpenCV should read the file
    image = cv2.imread(path, read_mode)           # actually load the image from disk
    if image is None:                             # cv2.imread returns None instead of an error if it fails
        raise FileNotFoundError(f"Could not read image: {path}")
    if not grayscale:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # OpenCV loads color images as BGR, we want RGB
    return image

In [ ]:
# Resizes a mask so it has the same width/height as reference_image, and scales its values from 0-255 to 0-1.
def resize_mask(mask, reference_image):
    target_size = (reference_image.shape[1], reference_image.shape[0])  # (width, height)
    if mask.shape[:2] == reference_image.shape[:2]:  # already the same shape? then skip resizing
        resized_mask = mask
    else:
        resized_mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)  # stretch/shrink to match
    resized_mask = resized_mask / 255.0  # turn 0-255 pixel values into 0.0-1.0
    return resized_mask

In [ ]:
# Blacks out every pixel that is outside the mask (mask value 0 = hidden, 1 = kept).
def apply_mask(image, mask):
    masked_image = image * mask[..., np.newaxis].astype(np.uint8)  # multiply each pixel by 0 or 1
    return masked_image

In [ ]:
# Plots one or more images. 1 image = 1 column, more than 1 image = 2 columns, with as many rows as needed.
def plot_images(images, titles=None, cmap=None):
    num_images = len(images)                        # how many images we were given
    num_cols = 2 if num_images > 1 else 1            # 2 columns if we have more than 1 image, else just 1
    num_rows = int(np.ceil(num_images / num_cols))   # enough rows to fit all images

    fig, axes = plt.subplots(num_rows, num_cols, figsize=(6 * num_cols, 6 * num_rows))
    axes = np.array(axes).reshape(-1)  # flatten so indexing works for any num_images

    for image_index, image in enumerate(images):     # draw each image into its own subplot
        axes[image_index].imshow(image, cmap=cmap)
        if titles is not None:
            axes[image_index].set_title(titles[image_index])
        axes[image_index].axis("off")                # hide axis ticks, we just want the picture

    for empty_index in range(num_images, len(axes)):  # hide unused axes when num_images is odd
        axes[empty_index].axis("off")

    plt.tight_layout()  # avoid titles/plots overlapping
    plt.show()            # display the figure

In [3]:
# Converts a color image to its Hue channel, in normal 0-360 degree units.
def hue_transform(image):
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)  # split image into Hue, Saturation, Value
    hue = hsv[..., 0]  # 0-179 range (uint8 image, so OpenCV halves the degrees)
    hue_degrees = hue.astype(np.float32) * 2  # convert back to normal 0-360 degrees
    return hue_degrees

In [1]:
# Finds the main hue "peaks" (common colors) inside the masked area of the image.
def calculate_hue_peaks(hue_degrees, region_mask, num_bins=90, prominence_ratio=0.01, min_width=1, rel_height=0.5):
    hue_degrees_in_region = hue_degrees[region_mask > 0]  # keep only hue values inside the mask
    counts, bin_edges = np.histogram(hue_degrees_in_region, bins=num_bins, range=(0, 360))  # pixels per hue bucket

    # a "peak" is a bump in the histogram = a common color
    peak_indices, peak_properties = find_peaks(
        counts, prominence=counts.max() * prominence_ratio, width=min_width, rel_height=rel_height
    )
    hue_peaks = bin_edges[peak_indices]  # turn peak bucket positions back into actual hue degree values

    return hue_peaks, hue_degrees_in_region

In [ ]:
# For every pixel, finds its closest hue peak, then counts pixels/percent/area per peak and saves it to a json file.
def calculate_habitat_areas(hue_degrees_in_region, hue_peaks, roi_pixels, total_area_km2,
                             peak_names, max_hue_distance, output_path="data/saved/habitat_areas.json"):

    total_region_pixels = np.sum(roi_pixels)
    hue_diff = np.abs(hue_degrees_in_region[:, None] - hue_peaks[None, :])  # distance from every pixel to every peak
    hue_distance = np.minimum(hue_diff, 360 - hue_diff)  # handle hue wraparound (0 and 360 are the same color)

    nearest_peak_index = np.argmin(hue_distance, axis=1)     # for each pixel, index of its closest peak
    nearest_peak_distance = np.min(hue_distance, axis=1)     # for each pixel, how far that closest peak is

    pixel_labels = hue_peaks[nearest_peak_index]              # each pixel's label = its closest peak's hue
    pixel_labels[nearest_peak_distance > max_hue_distance] = -1  # too far from any peak -> "uncategorised" (-1)

    category_areas = {}
    for peak in list(hue_peaks) + [-1]:              # go through every peak, plus "uncategorised"
        pixel_count = int(np.sum(pixel_labels == peak))         # how many pixels got this label
        category_name = "unknown" if peak == -1 else peak_names[peak]  # turn the hue number into a real category name

        if category_name not in category_areas:
            category_areas[category_name] = {"pixel_count": 0}
        category_areas[category_name]["pixel_count"] += pixel_count  # -1 and unmapped peaks fold into "unknown"

    for category_name, category_data in category_areas.items():  # now add percent and area for each category
        percent = 100 * category_data["pixel_count"] / total_region_pixels
        category_data["percent"] = round(percent, 2)
        category_data["area_km2"] = round(total_area_km2 * percent / 100, 2)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)  # make sure the destination folder (e.g. data/saved) exists

    with open(output_path, "w") as output_file:         # save the results as a json file
        json.dump(category_areas, output_file, indent=2)

    return category_areas